# KevRouter: Exploratory Data Analysis (EDA)

**Authors:** Blake DeHaas & Abhinav Dugar  
**Project:** Cost-Sensitive LLM Routing with Kev Decision Models  

This notebook explores the dataset from **LLMRouterBench** (`NPULH/LLMRouterBench`) to understand:
1. **Dataset Splits & Completeness:** 12,168 primary queries across 9 benchmarks + 278 held-out $\tau^2$-Bench queries.
2. **Destination Model Space:** Performance and pricing of the 13 destination models.
3. **Optimal Routing Labels:** Distribution of the ground-truth cheapest correct model across queries.
4. **Token Length Distributions:** Verifying the ~1,024 average token estimate for training compute and VRAM sizing.
5. **Routing Potential & Cost Savings:** Oracle Router vs. Best Single Model.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Paths
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
print(f"Processed Data Directory: {PROCESSED_DIR}")

## 1. Load Processed Dataset Splits

In [ ]:
def load_split(name):
    path = PROCESSED_DIR / f"{name}.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run scripts/download_data.py and scripts/build_dataset.py first.")
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    df = pd.DataFrame(records)
    df["split"] = name
    return df

df_train = load_split("train")
df_val = load_split("val")
df_test_id = load_split("test_id")
df_test_ood = load_split("test_ood")

print(f"Train records:   {len(df_train):>6,} (Target: 9,734)")
print(f"Val records:     {len(df_val):>6,} (Target: 1,217)")
print(f"Test ID records: {len(df_test_id):>6,} (Target: 1,217)")
print(f"Total Primary:   {len(df_train) + len(df_val) + len(df_test_id):>6,} (Target: 12,168)")
print(f"Test OOD (tau2): {len(df_test_ood):>6,} (Target: 278)")

df_primary = pd.concat([df_train, df_val, df_test_id], ignore_index=True)

## 2. Benchmark Query Distribution

In [ ]:
benchmark_counts = df_primary["dataset"].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=benchmark_counts.values, y=benchmark_counts.index, palette="viridis")
plt.title("Query Counts across 9 Primary Benchmarks", fontsize=14, fontweight="bold")
plt.xlabel("Number of Queries")
plt.ylabel("Benchmark")
for i, v in enumerate(benchmark_counts.values):
    plt.text(v + 15, i, f"{v:,}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Destination Models: Accuracy vs. Cost Frontier

In [ ]:
# Extract model accuracy and average cost
model_data = []
all_models = list(df_primary.iloc[0]["model_evals"].keys())

for m in all_models:
    scores = [r["model_evals"][m]["score"] for r in df_primary.to_dict("records")]
    costs = [r["model_evals"][m]["cost"] for r in df_primary.to_dict("records")]
    model_data.append({
        "model": m,
        "accuracy": np.mean(scores) * 100,
        "avg_cost_usd": np.mean(costs),
    })

df_models = pd.DataFrame(model_data).sort_values("accuracy", ascending=False)
display(df_models)

# Scatter plot: Accuracy vs. Cost
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_models, x="avg_cost_usd", y="accuracy", s=120, color="#1f77b4")
for _, row in df_models.iterrows():
    plt.annotate(row["model"], (row["avg_cost_usd"], row["accuracy"] + 0.5), fontsize=9)
plt.title("Destination Models: Accuracy vs. Average Cost per Query", fontsize=14, fontweight="bold")
plt.xlabel("Average Cost per Query ($ USD)")
plt.ylabel("Accuracy (%)")
plt.tight_layout()
plt.show()

## 4. Optimal Routing Target Class Distribution
Which destination models does the cost-sensitive objective select most frequently as the cheapest correct model?

In [ ]:
routing_share = df_primary["optimal_model"].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=routing_share.values, y=routing_share.index, palette="magma")
plt.title("Optimal Routing Target (Cheapest Correct Model) Share", fontsize=14, fontweight="bold")
plt.xlabel("Number of Queries Where Model is Ideal Route")
plt.ylabel("Destination Model")
for i, v in enumerate(routing_share.values):
    pct = (v / len(df_primary)) * 100
    plt.text(v + 10, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Token Length Distribution
Checking prompt token lengths to validate memory footprint and compute estimations.

In [ ]:
def extract_prompt_tokens(row):
    for m_info in row["model_evals"].values():
        if m_info.get("prompt_tokens") is not None:
            return int(m_info["prompt_tokens"])
    return len(row.get("prompt", "").split())

df_primary["prompt_tokens"] = df_primary.apply(extract_prompt_tokens, axis=1)

print(f"Mean prompt tokens:   {df_primary['prompt_tokens'].mean():.1f}")
print(f"Median prompt tokens: {df_primary['prompt_tokens'].median():.1f}")
print(f"95th percentile:      {df_primary['prompt_tokens'].quantile(0.95):.1f}")
print(f"Max prompt tokens:    {df_primary['prompt_tokens'].max():.1f}")

plt.figure(figsize=(10, 5))
sns.histplot(df_primary["prompt_tokens"], bins=50, kde=True, color="teal")
plt.axvline(1024, color="red", linestyle="--", label="Proposal Assumption (1,024 tokens)")
plt.axvline(df_primary["prompt_tokens"].mean(), color="orange", linestyle="-", label=f"Mean ({df_primary['prompt_tokens'].mean():.0f})")
plt.title("Distribution of Query Prompt Tokens", fontsize=14, fontweight="bold")
plt.xlabel("Prompt Tokens")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Sample Query & Decision Inspection

In [ ]:
sample = df_primary.sample(1, random_state=42).iloc[0]
print(f"Benchmark:     {sample['dataset']}")
print(f"Optimal Model: {sample['optimal_model']} (Cost: ${sample['optimal_cost']:.5f})")
print(f"Prompt snippet: {sample['prompt'][:300]}...")
print("\nDestination Model Evaluations:")
for m, ev in sample["model_evals"].items():
    correct_icon = "✓" if ev['score'] >= 1.0 else "✗"
    star = "★ IDEAL" if m == sample['optimal_model'] else ""
    print(f"  [{correct_icon}] {m:<25} | Score: {ev['score']:.1f} | Cost: ${ev['cost']:.5f} {star}")